# 02 - EDA Fraud Analysis: PaySim
Tac gia: Khai (TV2)

Phan tich pattern gian lan, mat can bang nhan va hanh vi so du.

In [ ]:
import pandas as pd
from pathlib import Path
CSV_PATH = Path('../data/raw/PS_20174392719_1491204439457_log.csv')
CHUNK = 200_000

## 1. Fraud theo loai giao dich

In [ ]:
fraud_by_type = {}
total_fraud = 0
for ch in pd.read_csv(CSV_PATH, chunksize=CHUNK):
    fr = ch[ch['isFraud']==1]
    total_fraud += len(fr)
    for t in fr['type'].unique():
        fraud_by_type[t] = fraud_by_type.get(t,0) + int((fr['type']==t).sum())
print('Fraud by type:', fraud_by_type)
print('Total fraud:', total_fraud)

Ket qua: Fraud chi o TRANSFER (4.097) va CASH_OUT (4.116). PAYMENT/DEBIT/CASH_IN = 0.

## 2. Full-drain pattern
8.024/8.213 (97.7 phan tram) giao dich fraud co oldbalanceOrg == amount va newbalanceOrig == 0.

In [ ]:
drain = 0
tf = 0
for ch in pd.read_csv(CSV_PATH, chunksize=CHUNK):
    fr = ch[ch['isFraud']==1]
    tf += len(fr)
    cond = (abs(fr['oldbalanceOrg']-fr['amount'])<0.01) & (fr['newbalanceOrig']==0)
    drain += int(cond.sum())
pct = round(drain*100.0/tf, 1)
print('Full-drain:', drain, '/', tf, '=', pct, 'phan tram')

## 3. isFlaggedFraud reliability
Chi 16 dong duoc danh dau isFlaggedFraud=1. Khong dung lam nhan ground truth.

In [ ]:
flagged = 0
for ch in pd.read_csv(CSV_PATH, chunksize=CHUNK):
    flagged += int((ch['isFlaggedFraud']==1).sum())
print('isFlaggedFraud=1:', flagged)

## 4. Ket luan
- IsHighRiskType = 1 cho TRANSFER, CASH_OUT
- BalanceDropOrig la feature manh nhat
- isFlaggedFraud khong dang tin cay
- Chi tiet: docs/data/data_insights.md